In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

# =========================
# 1. LOAD DATA
# =========================

product_path = "03_product_performance.csv"
bcg_path = "03_product_bcg_line.csv"

product = pd.read_csv(product_path)
bcg = pd.read_csv(bcg_path)

# Chuẩn hóa tên cột
product.columns = product.columns.str.strip().str.lower()
bcg.columns = bcg.columns.str.strip().str.lower()

# Nếu cột Color bị thành color thì ok, nếu tên khác thì kiểm tra
print("Product columns:")
print(product.columns.tolist())

print("\nBCG columns:")
print(bcg.columns.tolist())

# Chuẩn hóa text
text_cols_product = ["group_code", "group_name", "line_name", "product_code", "product_name", "color"]
for col in text_cols_product:
    if col in product.columns:
        product[col] = product[col].astype(str).str.strip()

text_cols_bcg = ["line_name", "group_name", "bcg_group"]
for col in text_cols_bcg:
    if col in bcg.columns:
        bcg[col] = bcg[col].astype(str).str.strip()

# Đổi các cột số về numeric
num_cols_product = [
    "order_count", "customer_count", "total_quantity",
    "total_revenue", "avg_unit_price", "revenue_per_unit"
]

for col in num_cols_product:
    if col in product.columns:
        product[col] = pd.to_numeric(product[col], errors="coerce")

num_cols_bcg = [
    "revenue_q1_2025", "revenue_q1_2026",
    "revenue_diff", "growth_pct",
    "median_revenue", "median_growth"
]

for col in num_cols_bcg:
    if col in bcg.columns:
        bcg[col] = pd.to_numeric(bcg[col], errors="coerce")

# Fill NA số bằng 0 nếu cần
product[num_cols_product] = product[num_cols_product].fillna(0)
bcg[num_cols_bcg] = bcg[num_cols_bcg].fillna(0)

print(product.head())
print(bcg.head())

Product columns:
['group_code', 'group_name', 'line_id_fk', 'line_name', 'product_code', 'product_name', 'color', 'order_count', 'customer_count', 'total_quantity', 'total_revenue', 'avg_unit_price', 'revenue_per_unit']

BCG columns:
['line_name', 'group_name', 'revenue_q1_2025', 'revenue_q1_2026', 'revenue_diff', 'growth_pct', 'median_revenue', 'median_growth', 'bcg_group']
   group_code    group_name  line_id_fk     line_name     product_code  \
0  CITYBIKE_P  Xe phổ thông        1.00  Xe 219-05-26  000207004000000   
1  CITYBIKE_P  Xe phổ thông        2.00     Xe 219-24  000208002003000   
2  CITYBIKE_P  Xe phổ thông        2.00     Xe 219-24  000208002007000   
3  CITYBIKE_P  Xe phổ thông        2.00     Xe 219-24  000208002023000   
4  CITYBIKE_P  Xe phổ thông        3.00     Xe 219-26  000209002003000   

                        product_name      color  order_count  customer_count  \
0        Xe đạp Thống Nhất 219-05-26  219-05-26          241             166   
1    Xe đạp Thống

In [ ]:
# =========================
# BASIC CLEANING
# =========================

# Nếu file có cột Color viết hoa/lệch format thì đưa về color
rename_map = {
    "Color": "color",
    "COLOR": "color",
    "colour": "color",
    "product color": "color"
}

product = product.rename(columns={k.lower(): v for k, v in rename_map.items() if k.lower() in product.columns})

# Chuẩn hóa text columns
text_cols = ["group_code", "group_name", "line_name", "product_code", "product_name", "color"]

for col in text_cols:
    if col in product.columns:
        product[col] = (
            product[col]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NULL": pd.NA})
        )

# Chuẩn hóa numeric columns
num_cols = [
    "order_count", "customer_count", "total_quantity",
    "total_revenue", "avg_unit_price", "revenue_per_unit"
]

for col in num_cols:
    if col in product.columns:
        product[col] = pd.to_numeric(product[col], errors="coerce").fillna(0)

product.isna().sum()

,0
group_code,90
group_name,90
line_id_fk,90
line_name,90
product_code,0
product_name,0
color,18
order_count,0
customer_count,0
total_quantity,0


In [ ]:
import re
import pandas as pd
import numpy as np

# =========================
# FINAL FIX FOR STRING "nan" / REAL NaN GROUP + LINE
# =========================

# 1. Convert các giá trị "nan" dạng string thành pd.NA
cols_to_clean_missing = ["group_code", "group_name", "line_name", "color"]

for col in cols_to_clean_missing:
    if col in product.columns:
        product[col] = (
            product[col]
            .astype("string")
            .str.strip()
            .replace({
                "": pd.NA,
                "nan": pd.NA,
                "NaN": pd.NA,
                "NAN": pd.NA,
                "none": pd.NA,
                "None": pd.NA,
                "NULL": pd.NA,
                "null": pd.NA,
                "<NA>": pd.NA,
                "Unknown": pd.NA,
                "UNKNOWN": pd.NA
            })
        )


def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()


# 2. Extract line_name tốt hơn từ product_name
def infer_line_name_v3(product_name):
    name = str(product_name).strip()

    # Bỏ prefix thương hiệu
    name = re.sub(r"(?i)^xe đạp thống nhất\s*", "", name).strip()

    # Bỏ các cụm không phải line
    name = re.sub(r"(?i)\bmàu\b", " ", name)
    name = re.sub(r"(?i)\bDA HP\b|\bD A HP\b|\bDA\b|\bHP\b|\bKyolic\b|\bBase\b", " ", name)

    # Bỏ màu ở cuối tên
    color_suffixes = [
        "Xanh da trời",
        "Xanh dương",
        "Xanh biển",
        "Xanh Santorini",
        "Xanh Mint",
        "Xanh lá",
        "Hồng dạ quang",
        "Tím dạ quang",
        "Vàng cánh gián",
        "Đỏ đậm",
        "Café/nâu",
        "Cafe/nâu",
        "Đen",
        "Trắng",
        "Đỏ",
        "Xanh",
        "Hồng",
        "Tím",
        "Vàng",
        "Cam",
        "Ghi",
        "Xám",
        "Bạc",
        "Nâu",
        "Kem",
        "Coban",
        "Be",
        "Da"
    ]

    for color in sorted(color_suffixes, key=len, reverse=True):
        name = re.sub(rf"(?i)\s+{re.escape(color)}\s*$", "", name).strip()

    # Chuẩn hóa một số dòng cụ thể
    name_lower = name.lower()

    if re.search(r"road\s*rpd\s*700c\s*v5", name_lower):
        return "Road RPD 700C V5"

    if re.search(r"mtb\s*spd\s*27\.?\s*5", name_lower) or re.search(r"mtb\s*spd\s*27,\s*5", name_lower):
        return "MTB SPD 27.5"

    if re.search(r"\bte\s*16\s*03\b", name_lower):
        return "TE 16 03"

    if re.search(r"\bte\s*16\b", name_lower):
        return "TE 16"

    if re.search(r"\bte\s*20\b", name_lower):
        return "TE 20"

    if re.search(r"unite\s*27\.?\s*5", name_lower) or re.search(r"unite\s*27,\s*5", name_lower):
        return "Unite 27.5"

    if re.search(r"grx\s*2\.0", name_lower):
        return "GRX 2.0"

    if re.search(r"grx\s*at", name_lower):
        return "GRX AT 27.5 2.0"

    if re.search(r"\brex\b", name_lower):
        return "REX"

    if re.search(r"\bslx\b", name_lower):
        return "SLX 27.5"

    if re.search(r"the\s*flash|flash", name_lower):
        return "The Flash 27"

    if re.search(r"\bms\s*27", name_lower):
        return "MS 27.5"

    if re.search(r"\bsk\s*20\b", name_lower):
        return "SK 20"

    if re.search(r"\bld\s*24", name_lower):
        return "LD 24"

    if re.search(r"\bld\s*26", name_lower):
        return "LD 26"

    if re.search(r"new\s*24", name_lower):
        return "Xe New 24"

    if re.search(r"new\s*26", name_lower):
        return "Xe New 26"

    if re.search(r"\bnữ\b|\bnu\b", name_lower):
        return "Xe Nữ"

    if re.search(r"\bnam\b", name_lower):
        return "Xe Nam"

    # Clean khoảng trắng
    name = re.sub(r"\s+", " ", name).strip()

    return name if name else pd.NA


# 3. Infer group từ product_name + line_name
def infer_group_name_v3(row):
    current = row.get("group_name", pd.NA)

    if pd.notna(current):
        current_str = str(current).strip()
        if current_str.lower() not in ["", "nan", "none", "unknown", "<na>"]:
            return current_str

    product_name = norm_text(row.get("product_name", ""))
    line_name = norm_text(row.get("line_name", ""))
    text = product_name + " " + line_name

    # Xe trẻ em
    kid_patterns = [
        r"we bare bears",
        r"tom\s*&\s*jerry",
        r"tom and jerry",
        r"batwheels",
        r"bubbles",
        r"te\s*16",
        r"te\s*20",
        r"\bte\b",
        r"kid",
        r"trẻ em",
        r"tre em"
    ]

    if any(re.search(p, text, flags=re.IGNORECASE) for p in kid_patterns):
        return "Xe trẻ em"

    # Xe phổ thông
    city_patterns = [
        r"\bnam\b",
        r"\bnữ\b",
        r"\bnu\b",
        r"\bld\b",
        r"new\s*24",
        r"new\s*26",
        r"\bgn\b",
        r"city",
        r"phổ thông",
        r"pho thong",
        r"\bms\s*27",
        r"\bsk\s*20"
    ]

    if any(re.search(p, text, flags=re.IGNORECASE) for p in city_patterns):
        return "Xe phổ thông"

    # Xe thể thao nhôm
    alloy_patterns = [
        r"road",
        r"rpd",
        r"700c",
        r"grx",
        r"rex",
        r"unite",
        r"touring",
        r"blade",
        r"alloy",
        r"nhôm",
        r"nhom"
    ]

    if any(re.search(p, text, flags=re.IGNORECASE) for p in alloy_patterns):
        return "Xe thể thao nhôm"

    # Xe thể thao thép
    steel_patterns = [
        r"mtb",
        r"spd",
        r"slx",
        r"flash",
        r"sport",
        r"highway",
        r"shimano",
        r"thể thao",
        r"the thao"
    ]

    if any(re.search(p, text, flags=re.IGNORECASE) for p in steel_patterns):
        return "Xe thể thao thép"

    return "Unknown"


# 4. Fill line_name nếu đang thiếu
missing_line_mask = product["line_name"].isna()

product.loc[missing_line_mask, "line_name"] = product.loc[
    missing_line_mask, "product_name"
].apply(infer_line_name_v3)

# 5. Clean lại line_name đang có nhưng bị dính màu
product["line_name"] = product["line_name"].apply(
    lambda x: infer_line_name_v3(x) if pd.notna(x) else x
)

# 6. Fill group_name/group_code
product["group_name"] = product.apply(infer_group_name_v3, axis=1)

group_code_map = {
    "Xe phổ thông": "CITYBIKE_P",
    "Xe trẻ em nhóm 1": "KIDBIKE",
    "Xe trẻ em nhóm 2": "KIDBIKE",
    "Xe trẻ em": "KIDBIKE",
    "Xe thể thao nhôm": "SPORTBIKE_ALLOY",
    "Xe thể thao thép": "SPORTBIKE_STEEL",
    "Unknown": "UNKNOWN"
}

product["group_code"] = product["group_name"].map(group_code_map).fillna("UNKNOWN")


# 7. Force fix một số pattern cụ thể còn sót
text_all = (
    product["product_name"].astype(str).str.lower() + " " +
    product["line_name"].astype(str).str.lower()
)

# Road RPD 700C V5
mask_road = text_all.str.contains(r"road|rpd|700c", regex=True, na=False)
product.loc[mask_road, "group_name"] = "Xe thể thao nhôm"
product.loc[mask_road, "group_code"] = "SPORTBIKE_ALLOY"
product.loc[mask_road, "line_name"] = "Road RPD 700C V5"

# MTB SPD 27.5
mask_mtb_spd = text_all.str.contains(r"mtb|spd", regex=True, na=False)
product.loc[mask_mtb_spd, "group_name"] = "Xe thể thao thép"
product.loc[mask_mtb_spd, "group_code"] = "SPORTBIKE_STEEL"
product.loc[mask_mtb_spd, "line_name"] = "MTB SPD 27.5"

# TE 16 / TE 20
mask_te16 = text_all.str.contains(r"\bte\s*16\b", regex=True, na=False)
product.loc[mask_te16, "group_name"] = "Xe trẻ em"
product.loc[mask_te16, "group_code"] = "KIDBIKE"
product.loc[mask_te16, "line_name"] = "TE 16"

mask_te20 = text_all.str.contains(r"\bte\s*20\b", regex=True, na=False)
product.loc[mask_te20, "group_name"] = "Xe trẻ em"
product.loc[mask_te20, "group_code"] = "KIDBIKE"
product.loc[mask_te20, "line_name"] = "TE 20"

# Unite 27.5
mask_unite = text_all.str.contains(r"unite", regex=True, na=False)
product.loc[mask_unite, "group_name"] = "Xe thể thao nhôm"
product.loc[mask_unite, "group_code"] = "SPORTBIKE_ALLOY"
product.loc[mask_unite, "line_name"] = "Unite 27.5"


# 8. Final check
still_missing = product[
    product["group_name"].isna() |
    product["group_code"].isna() |
    product["line_name"].isna() |
    product["group_name"].astype(str).str.lower().isin(["nan", "unknown", "none", "<na>"]) |
    product["group_code"].astype(str).str.lower().isin(["nan", "unknown", "none", "<na>"])
]

print("Remaining problematic rows:", len(still_missing))

display(
    still_missing[
        [
            "group_code", "group_name", "line_name",
            "product_code", "product_name", "color",
            "total_quantity", "total_revenue"
        ]
    ].sort_values("product_name").head(100)
)

print("\nCheck fixed rows:")
check_mask = product["product_name"].astype(str).str.lower().str.contains(
    "road|rpd|700c|mtb|spd|te 16|te 20|unite|grx|rex|slx|flash",
    regex=True,
    na=False
)

display(
    product[check_mask][
        [
            "group_code", "group_name", "line_name",
            "product_code", "product_name", "color",
            "total_quantity", "total_revenue"
        ]
    ].sort_values(["group_name", "line_name", "product_name"])
)

Remaining problematic rows: 11


,group_code,group_name,line_name,product_code,product_name,color,total_quantity,total_revenue
209,UNKNOWN,Unknown,MS Superman 27.5 DC,000233003022001,Xe đạp Thống Nhất MS Superman 27.5 DC xanh,xanh,21.00,"48,264,103.00"
189,UNKNOWN,Unknown,Neo 20-03,000123002007000,Xe đạp Thống Nhất Neo 20-03 Coban,Coban,22.00,"25,055,556.00"
227,UNKNOWN,Unknown,SK 24,000337002008000,Xe đạp Thống Nhất SK 24 ghi,ghi,456.00,"737,955,472.00"
228,UNKNOWN,Unknown,SK 24,000337002022000,Xe đạp Thống Nhất SK 24 xanh,xanh,305.00,"504,413,285.00"
226,UNKNOWN,Unknown,SK 24,000337002001000,Xe đạp Thống Nhất SK 24 đen,đen,681.00,"1,098,703,209.00"
219,UNKNOWN,Unknown,Super 26 S,000333002008000,Xe đạp Thống Nhất Super 26 S ghi,ghi,390.00,"811,817,493.00"
220,UNKNOWN,Unknown,Super 26 S,000333002008003,Xe đạp Thống Nhất Super 26 S ghi DA HP,HP,52.00,"134,814,815.00"
221,UNKNOWN,Unknown,Super 26 S,000333002022002,Xe đạp Thống Nhất Super 26 S xanh,xanh,308.00,"626,194,374.00"
222,UNKNOWN,Unknown,Super 26 S,000333002022003,Xe đạp Thống Nhất Super 26 S xanh DA HP,HP,52.00,"134,814,815.00"
217,UNKNOWN,Unknown,Super 26 S,000333002001002,Xe đạp Thống Nhất Super 26 S đen,đen,152.00,"338,283,272.00"



Check fixed rows:


,group_code,group_name,line_name,product_code,product_name,color,total_quantity,total_revenue
251,SPORTBIKE_ALLOY,Xe thể thao nhôm,GRX 2.0,1010030010260000,Xe đạp Thống Nhất GRX 2.0 Be,Be,1.00,"6,840,277.00"
250,SPORTBIKE_ALLOY,Xe thể thao nhôm,GRX 2.0,1010030010220000,Xe đạp Thống Nhất GRX 2.0 Xanh dương,Xanh dương,9.00,"66,224,531.00"
249,SPORTBIKE_ALLOY,Xe thể thao nhôm,GRX 2.0,1010030010130000,Xe đạp Thống Nhất GRX 2.0 Xanh lá,Xanh lá,32.00,"211,732,850.00"
248,SPORTBIKE_ALLOY,Xe thể thao nhôm,GRX AT 27.5 2.0,1010020000220000,"Xe đạp Thống Nhất GRX AT 27, 5_2.0_15 Xanh",<NA>,8.00,"16,560,000.00"
124,SPORTBIKE_ALLOY,Xe thể thao nhôm,GRX AT 27.5 2.0,1010020000050000,"Xe đạp Thống Nhất GRX AT 27,5_2.0_15 Cam",Cam,15.00,"58,813,703.00"
...,...,...,...,...,...,...,...,...
262,SPORTBIKE_STEEL,Xe thể thao thép,SLX 27.5,TP0099.0000568,"Xe đạp Thống Nhất SLX 27, 5 - 01",<NA>,2.00,"2,200,000.00"
257,SPORTBIKE_STEEL,Xe thể thao thép,The Flash 27,TP0017.06.27.04,Xe đạp Thống Nhất The flash 27 01,<NA>,3.00,"3,450,000.00"
258,KIDBIKE,Xe trẻ em,TE 16,TP0022.02.16.00,Xe đạp Thống Nhất TE 16 02,<NA>,5.00,"4,500,000.00"
259,KIDBIKE,Xe trẻ em,TE 16,TP0022.03.16.00,Xe đạp Thống Nhất TE 16 03,<NA>,2.00,"1,900,000.00"


In [ ]:
# =========================
# FINAL MANUAL FIX: Neo 20-03 / SK 24 / Super 26 S
# =========================

text_all = (
    product["product_name"].astype(str).str.lower() + " " +
    product["line_name"].astype(str).str.lower()
)

# 1. Neo 20-03 -> Xe trẻ em
neo20_mask = text_all.str.contains(r"neo\s*20[-\s]*03|neo\s*20", regex=True, na=False)

product.loc[neo20_mask, "group_name"] = "Xe trẻ em"
product.loc[neo20_mask, "group_code"] = "KIDBIKE"
product.loc[neo20_mask, "line_name"] = "Neo 20-03"


# 2. SK 24 -> Xe phổ thông
sk24_mask = text_all.str.contains(r"\bsk\s*24\b", regex=True, na=False)

product.loc[sk24_mask, "group_name"] = "Xe phổ thông"
product.loc[sk24_mask, "group_code"] = "CITYBIKE_P"
product.loc[sk24_mask, "line_name"] = "SK 24"


# 3. Super 26 S -> Xe thể thao thép
super26s_mask = text_all.str.contains(r"super\s*26\s*s", regex=True, na=False)

product.loc[super26s_mask, "group_name"] = "Xe thể thao thép"
product.loc[super26s_mask, "group_code"] = "SPORTBIKE_STEEL"
product.loc[super26s_mask, "line_name"] = "Super 26 S"


# 4. MS Superman 27.5 DC -> Xe thể thao thép
ms_superman_mask = text_all.str.contains(r"ms\s*superman\s*27\.?5\s*dc", regex=True, na=False)

product.loc[ms_superman_mask, "group_name"] = "Xe thể thao thép"
product.loc[ms_superman_mask, "group_code"] = "SPORTBIKE_STEEL"
product.loc[ms_superman_mask, "line_name"] = "MS Superman 27.5 DC"
product.loc[ms_superman_mask, "color"] = "Xanh"
product.loc[ms_superman_mask, "color_clean"] = "Xanh"


# =========================
# CHECK AGAIN
# =========================

check_final_mask = neo20_mask | sk24_mask | super26s_mask | ms_superman_mask

print("Remaining problematic rows:", len(product[check_final_mask]))

display(
    product[check_final_mask][
        [
            "group_code", "group_name", "line_name",
            "product_code", "product_name", "color",
            "total_quantity", "total_revenue"
        ]
    ].sort_values(["group_name", "line_name", "product_name"])
)

# Check còn Unknown không
remaining_unknown = product[
    product["group_name"].astype(str).str.lower().isin(["unknown", "nan", "none", "<na>"]) |
    product["group_code"].astype(str).str.lower().isin(["unknown", "nan", "none", "<na>"]) |
    product["group_name"].isna() |
    product["group_code"].isna()
]

print("Remaining Unknown rows:", len(remaining_unknown))

display(
    remaining_unknown[
        [
            "group_code", "group_name", "line_name",
            "product_code", "product_name", "color",
            "total_quantity", "total_revenue"
        ]
    ].sort_values("product_name")
)

Remaining problematic rows: 23


,group_code,group_name,line_name,product_code,product_name,color,total_quantity,total_revenue
227,CITYBIKE_P,Xe phổ thông,SK 24,000337002008000,Xe đạp Thống Nhất SK 24 ghi,ghi,456.00,"737,955,472.00"
228,CITYBIKE_P,Xe phổ thông,SK 24,000337002022000,Xe đạp Thống Nhất SK 24 xanh,xanh,305.00,"504,413,285.00"
226,CITYBIKE_P,Xe phổ thông,SK 24,000337002001000,Xe đạp Thống Nhất SK 24 đen,đen,681.00,"1,098,703,209.00"
209,SPORTBIKE_STEEL,Xe thể thao thép,MS Superman 27.5 DC,000233003022001,Xe đạp Thống Nhất MS Superman 27.5 DC xanh,Xanh,21.00,"48,264,103.00"
219,SPORTBIKE_STEEL,Xe thể thao thép,Super 26 S,000333002008000,Xe đạp Thống Nhất Super 26 S ghi,ghi,390.00,"811,817,493.00"
220,SPORTBIKE_STEEL,Xe thể thao thép,Super 26 S,000333002008003,Xe đạp Thống Nhất Super 26 S ghi DA HP,HP,52.00,"134,814,815.00"
221,SPORTBIKE_STEEL,Xe thể thao thép,Super 26 S,000333002022002,Xe đạp Thống Nhất Super 26 S xanh,xanh,308.00,"626,194,374.00"
222,SPORTBIKE_STEEL,Xe thể thao thép,Super 26 S,000333002022003,Xe đạp Thống Nhất Super 26 S xanh DA HP,HP,52.00,"134,814,815.00"
217,SPORTBIKE_STEEL,Xe thể thao thép,Super 26 S,000333002001002,Xe đạp Thống Nhất Super 26 S đen,đen,152.00,"338,283,272.00"
218,SPORTBIKE_STEEL,Xe thể thao thép,Super 26 S,000333002001003,Xe đạp Thống Nhất Super 26 S đen DA HP,HP,52.00,"134,814,815.00"


Remaining Unknown rows: 0


,group_code,group_name,line_name,product_code,product_name,color,total_quantity,total_revenue


In [ ]:
# =========================
# QUALITY CHECK AFTER CLEANING
# =========================

print("Missing values after cleaning:")
print(product[["group_code", "group_name", "line_name", "product_code", "product_name", "color"]].isna().sum())

print("\nGroup distribution:")
display(product["group_name"].value_counts(dropna=False))

print("\nColor distribution:")
display(product["color"].value_counts(dropna=False))

print("\nRows with Unknown group:")
display(product[product["group_name"].eq("Unknown")].head(20))

print("\nRows with Unknown color:")
display(product[product["color"].eq("NA")].head(20))

Missing values after cleaning:
group_code       0
group_name       0
line_name        0
product_code     0
product_name     0
color           18
dtype: int64

Group distribution:


,count
group_name,
Xe phổ thông,86
Xe thể thao thép,76
Xe trẻ em,33
Xe thể thao nhôm,31
Xe trẻ em nhóm 2,23
Xe trẻ em nhóm 1,16



Color distribution:


,count
color,
Hồng,22
Ghi,22
Đen,20
<NA>,18
Xanh,16
Xanh dương,15
Coban,12
Trắng,12
Kem,11



Rows with Unknown group:


,group_code,group_name,line_id_fk,line_name,product_code,product_name,color,order_count,customer_count,total_quantity,total_revenue,avg_unit_price,revenue_per_unit,color_clean



Rows with Unknown color:


,group_code,group_name,line_id_fk,line_name,product_code,product_name,color,order_count,customer_count,total_quantity,total_revenue,avg_unit_price,revenue_per_unit,color_clean


In [ ]:
product.to_csv("/content/03_product_performance_clean.csv", index=False, encoding="utf-8-sig")
print("Saved clean file: /content/03_product_performance_clean.csv")

Saved clean file: /content/03_product_performance_clean.csv


In [ ]:
import pandas as pd
import re
import unicodedata
import numpy as np

INPUT_PATH = "03_product_performance_clean.csv"
df = pd.read_csv(INPUT_PATH, dtype={"product_code": str})

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Danh sách màu nên để cụm dài trước cụm ngắn
COLOR_PATTERNS = [
    ("xanh da trời", "Xanh da trời"),
    ("xanh nước biển", "Xanh nước biển"),
    ("xanh dương", "Xanh dương"),
    ("xanh santorini", "Xanh Santorini"),
    ("xanh mint", "Xanh mint"),
    ("pastel xanh", "Pastel Xanh"),
    ("xanh tím", "Xanh tím"),
    ("xanh ngọc", "Xanh ngọc"),

    ("vàng chanh", "Vàng chanh"),
    ("vàng cánh gián", "Vàng cánh gián"),

    ("hồng dạ quang", "Hồng dạ quang"),
    ("tím dạ quang", "Tím dạ quang"),

    ("đỏ tươi", "Đỏ tươi"),
    ("đỏ đun", "Đỏ đun"),
    ("đỏ đậm", "Đỏ đậm"),

    ("đen bóng", "Đen bóng"),
    ("đen mờ", "Đen mờ"),

    ("café/nâu", "Café/nâu"),
    ("cafe/nâu", "Café/nâu"),

    ("coban", "Coban"),
    ("trắng", "Trắng"),
    ("đen", "Đen"),
    ("đỏ", "Đỏ"),
    ("xanh", "Xanh"),
    ("vàng", "Vàng"),
    ("hồng", "Hồng"),
    ("ghi", "Ghi"),
    ("kem", "Kem"),
    ("cam", "Cam"),
    ("tím", "Tím"),
    ("nâu", "Nâu"),
    ("bạc", "Bạc"),
    ("rêu", "Rêu"),
    ("mint", "Mint"),
    ("ngọc", "Xanh ngọc"),
]

COLOR_REGEX = "|".join(re.escape(x[0]) for x in COLOR_PATTERNS)
COLOR_MAP = dict(COLOR_PATTERNS)

def standardize_color_text(color):
    color = normalize_text(color).lower()

    if color in COLOR_MAP:
        return COLOR_MAP[color]

    # Sửa các màu bị cắt cụt từ logic cũ
    short_fix = {
        "hp": np.nan,
        "bat man": np.nan,
        "superman": np.nan,
        "wonderwoman": np.nan,
        "blackpink": np.nan,
        "trời": "Xanh da trời",
        "chanh": "Vàng chanh",
        "quang": np.nan,
        "gián": "Vàng cánh gián",
        "đậm": "Đỏ đậm",
        "mờ": "Đen mờ",
        "bóng": "Đen bóng",
    }

    return short_fix.get(color, normalize_text(color) if color else np.nan)

def extract_tem_color(product_name):
    name = normalize_text(product_name).lower()

    # Bắt cả dạng "(tem tím)" và "Tem Xanh da trời"
    m = re.search(r"\(?\btem\s+(" + COLOR_REGEX + r")\)?", name, flags=re.IGNORECASE)
    if m:
        raw = m.group(1).lower()
        return COLOR_MAP.get(raw, raw.title())

    return np.nan

def extract_main_color(product_name, old_color=None):
    name_original = normalize_text(product_name)
    name = name_original.lower()

    # Bỏ thương hiệu để đỡ nhiễu
    name = name.replace("xe đạp thống nhất", " ")

    # Bỏ hậu tố kỹ thuật / quyền sở hữu không phải màu
    name = re.sub(r"\bda\s*hp\b", " ", name, flags=re.IGNORECASE)

    # Bỏ phần sau dấu "-" như "- Bat man", "- Superman", "- Wonderwoman"
    name = re.sub(r"\s*-\s*(bat man|superman|wonderwoman|blackpink).*", " ", name, flags=re.IGNORECASE)

    # Lấy màu tem riêng, rồi bỏ cụm tem khỏi tên để ưu tiên màu chính
    tem_color = extract_tem_color(name_original)
    name_without_tem = re.sub(
        r"\(?\btem\s+(" + COLOR_REGEX + r")\)?",
        " ",
        name,
        flags=re.IGNORECASE
    )

    # Tìm màu chính trong phần tên đã bỏ tem
    matches = []
    for raw, standard in COLOR_PATTERNS:
        for m in re.finditer(r"(?<![a-zA-ZÀ-ỹ])" + re.escape(raw) + r"(?![a-zA-ZÀ-ỹ])", name_without_tem, flags=re.IGNORECASE):
            matches.append((m.start(), len(raw), standard))

    if matches:
        # Lấy match xuất hiện muộn nhất; nếu trùng vị trí thì lấy cụm dài hơn
        matches = sorted(matches, key=lambda x: (x[0], x[1]))
        return matches[-1][2]

    # Nếu không có màu chính nhưng có tem màu, dùng tem màu làm fallback
    # Ví dụ: "MTB 20-04 Tem đỏ" không có màu nào ngoài tem đỏ.
    if pd.notna(tem_color):
        return tem_color

    # Cuối cùng mới fallback về cột color cũ
    return standardize_color_text(old_color)

# Tạo thêm 2 cột để kiểm tra, chưa overwrite ngay
df["tem_color"] = df["product_name"].apply(extract_tem_color)
df["color_fixed"] = df.apply(
    lambda row: extract_main_color(row["product_name"], row.get("color", np.nan)),
    axis=1
)

# Nếu muốn dùng luôn màu đã sửa làm màu chính:
df["color_clean"] = df["color_fixed"]

# Check các dòng có màu thay đổi
changed = df[
    df["color"].astype(str).str.strip().str.lower()
    != df["color_fixed"].astype(str).str.strip().str.lower()
][[
    "product_code",
    "product_name",
    "color",
    "tem_color",
    "color_fixed",
    "color_clean"
]]



print("Number of rows with changed color:", len(changed))
display(changed)

# Xuất lại file clean mới
OUTPUT_PATH = "03_product_performance_clean_color_fixed.csv"
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved to:", OUTPUT_PATH)

Number of rows with changed color: 55


,product_code,product_name,color,tem_color,color_fixed,color_clean
3,000208002023000,Xe đạp Thống Nhất 219-24 Xanh Tím,Xanh Tím,NaN,Tím,Tím
6,000209002025000,Xe đạp Thống Nhất 219-26 Xanh Tím,Xanh Tím,NaN,Tím,Tím
32,000226003015000,Xe đạp Thống Nhất LD 24-01_2023 Café/nâu,Café/nâu,NaN,Nâu,Nâu
48,000224002015000,Xe đạp Thống Nhất New 24 Café/nâu,Café/nâu,NaN,Nâu,Nâu
51,000224002026000,Xe đạp Thống Nhất New 24 Xanh mint,Xanh mint,NaN,Mint,Mint
53,000225002002001,Xe đạp Thống Nhất New 26 Trắng DA HP,HP,NaN,Trắng,Trắng
55,000225002004001,Xe đạp Thống Nhất New 26 màu đỏ DA HP,HP,NaN,Đỏ,Đỏ
56,000225002007001,Xe đạp Thống Nhất New 26 Xanh DA HP,HP,NaN,Xanh,Xanh
58,000225002015000,Xe đạp Thống Nhất New 26 Café/nâu,Café/nâu,NaN,Nâu,Nâu
59,000225002015001,Xe đạp Thống Nhất New 26 Café/nâu DA HP,HP,NaN,Nâu,Nâu


Saved to: 03_product_performance_clean_color_fixed.csv


In [ ]:
# =========================
# 2. CATEGORY LEVEL ANALYSIS
# =========================

category_summary = (
    product
    .groupby(["group_code", "group_name"], as_index=False)
    .agg(
        total_revenue=("total_revenue", "sum"),
        total_quantity=("total_quantity", "sum"),
        order_count=("order_count", "sum"),
        customer_count=("customer_count", "sum"),
        avg_unit_price=("avg_unit_price", "mean"),
        revenue_per_unit=("revenue_per_unit", "mean"),
        sku_count=("product_code", "nunique"),
        line_count=("line_name", "nunique")
    )
)

category_summary["revenue_share_pct"] = (
    category_summary["total_revenue"] / category_summary["total_revenue"].sum() * 100
)

category_summary = category_summary.sort_values("total_revenue", ascending=False)

category_summary

,group_code,group_name,total_revenue,total_quantity,order_count,customer_count,avg_unit_price,revenue_per_unit,sku_count,line_count,revenue_share_pct
0,CITYBIKE_P,Xe phổ thông,"64,631,427,858.00","42,806.00",13530,8080,"1,688,602.34","1,686,872.15",86,26,59.05
5,SPORTBIKE_STEEL,Xe thể thao thép,"17,862,671,015.00","9,147.00",3525,2418,"1,988,291.02","1,981,806.92",76,14,16.32
2,KIDBIKE,Xe trẻ em nhóm 1,"10,131,262,145.00","8,111.00",3053,1885,"1,287,849.19","1,283,586.48",16,5,9.26
4,SPORTBIKE_ALLOY,Xe thể thao nhôm,"7,535,145,771.00","3,048.00",1289,820,"3,366,253.86","3,321,222.89",31,5,6.88
3,KIDBIKE,Xe trẻ em nhóm 2,"5,572,538,786.00","6,204.00",2115,1316,"1,029,452.00","1,012,254.71",23,14,5.09
1,KIDBIKE,Xe trẻ em,"3,712,115,864.00","2,830.00",1152,731,"1,220,013.87","1,203,016.70",33,10,3.39


In [ ]:
# =========================
# 3. LINE LEVEL ANALYSIS
# =========================

line_summary = (
    product
    .groupby(["group_name", "line_name"], as_index=False)
    .agg(
        total_revenue=("total_revenue", "sum"),
        total_quantity=("total_quantity", "sum"),
        order_count=("order_count", "sum"),
        customer_count=("customer_count", "sum"),
        avg_unit_price=("avg_unit_price", "mean"),
        sku_count=("product_code", "nunique"),
        color_count=("color", "nunique")
    )
)

line_summary["revenue_share_pct"] = (
    line_summary["total_revenue"] / line_summary["total_revenue"].sum() * 100
)

top10_lines_revenue = line_summary.sort_values("total_revenue", ascending=False).head(10)

top10_lines_revenue

,group_name,line_name,total_revenue,total_quantity,order_count,customer_count,avg_unit_price,sku_count,color_count,revenue_share_pct
23,Xe phổ thông,Xe New 26,"15,215,086,903.00","10,127.00",2933,1562,"1,650,859.05",13,9,13.90
32,Xe thể thao thép,MTB SPD 27.5,"12,157,413,136.00","6,217.00",2359,1582,"2,006,425.28",40,16,11.11
22,Xe phổ thông,Xe New 24,"9,443,952,289.00","6,790.00",2250,1388,"1,362,405.80",8,8,8.63
5,Xe phổ thông,LD 26,"7,776,161,256.00","4,438.00",1472,734,"1,796,061.07",5,5,7.11
29,Xe thể thao nhôm,Road RPD 700C V5,"6,629,967,284.00","2,693.00",1170,743,"2,553,169.65",16,7,6.06
4,Xe phổ thông,LD 24,"6,033,663,090.00","3,930.00",1302,715,"1,684,050.77",7,5,5.51
17,Xe phổ thông,Xe GN 06-26 2.0,"5,503,035,891.00","3,625.00",1160,649,"1,534,983.34",3,3,5.03
58,Xe trẻ em nhóm 1,Xe Puppy 20,"5,208,070,761.00","4,257.00",1523,897,"1,236,157.74",4,4,4.76
57,Xe trẻ em nhóm 1,Xe GN 06 20 2024,"3,109,181,192.00","2,624.00",1002,638,"1,192,870.30",4,4,2.84
15,Xe phổ thông,Xe GN 06-24 2.0,"2,713,862,635.00","1,886.00",705,473,"1,431,381.83",3,3,2.48
